# IoT Road Hazard Robot: YOLOv8 Training & Export in Google Colab

This notebook trains a custom YOLOv8 model for road hazard detection (potholes, cracks, animal road obstacles) and exports it to ONNX / TFLite for deployment to the Raspberry Pi / Jetson edge robot platform.

### Pipeline Steps:
1. Check GPU environment
2. Install Ultralytics and dependencies
3. Download / Prepare Road Hazard Dataset
4. Train YOLOv8n with road surface augmentations
5. Evaluate Model Metrics (mAP@0.5, Precision, Recall)
6. Export to ONNX / TFLite for edge deployment
7. Download weights

## 1. Verify GPU Hardware Acceleration

In [ ]:
!nvidia-smi

## 2. Install Ultralytics and Dependencies

In [ ]:
%pip install ultralytics onnx onnxruntime albumentations

## 3. Prepare Dataset & data.yaml
Classes:
- 0: pothole
- 1: crack
- 2: animal

In [ ]:
import yaml
from pathlib import Path

# Create data.yaml
data_config = {
    'path': '/content/dataset',
    'train': 'train/images',
    'val': 'val/images',
    'test': 'test/images',
    'names': {
        0: 'pothole',
        1: 'crack',
        2: 'animal'
    },
    'nc': 3
}

Path('/content/dataset').mkdir(parents=True, exist_ok=True)
with open('/content/dataset/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print('data.yaml generated successfully!')

## 4. Train YOLOv8n Model with Road Augmentations

In [ ]:
from ultralytics import YOLO

# Load pretrained YOLOv8 Nano
model = YOLO('yolov8n.pt')

# Train with road surface tailored augmentations
results = model.train(
    data='/content/dataset/data.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    optimizer='Adam',
    lr0=0.001,
    device=0,
    project='roadbot_training',
    name='yolov8n_road_hazards',
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=10.0,
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    fliplr=0.5,
    save=True
)

## 5. Evaluate Model on Test Split

In [ ]:
# Run validation
metrics = model.val(data='/content/dataset/data.yaml', split='test')

print(f'Mean Precision:    {metrics.box.mp:.4f}')
print(f'Mean Recall:       {metrics.box.mr:.4f}')
print(f'mAP @ 0.5:         {metrics.box.map50:.4f}')
print(f'mAP @ 0.5:0.95:    {metrics.box.map:.4f}')

## 6. Export to Edge Deployment Formats (ONNX & TFLite)

In [ ]:
# Export to ONNX
onnx_path = model.export(format='onnx', imgsz=640, dynamic=True)
print(f'ONNX Export Path: {onnx_path}')

# Export to TFLite (optional for edge TPU)
try:
    tflite_path = model.export(format='tflite', imgsz=640)
    print(f'TFLite Export Path: {tflite_path}')
except Exception as e:
    print(f'TFLite export error: {e}')

## 7. Download Trained Weights to Local Machine

In [ ]:
from google.colab import files

best_weights = 'roadbot_training/yolov8n_road_hazards/weights/best.pt'
files.download(best_weights)
files.download(best_weights.replace('.pt', '.onnx'))